<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2023/Day_13_Point_of_Incidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''#.##..##.
..#.##.#.
##......#
##......#
..#.##.#.
..##..##.
#.#.##.#.

#...##..#
#....#..#
..##..###
#####.##.
#####.##.
..##..###
#....#..#'''

with open('13.txt') as f: puzzle = f.read()

puzzle = puzzle.replace('.', '0').replace('#', '1')
puzzle = puzzle.split('\n\n')
puzzle = [pattern.split('\n') for pattern in puzzle]
puzzle = [[list(line) for line in pattern] for pattern in puzzle]
puzzle

import numpy as np
puzzle = [np.array(pattern, dtype = int) for pattern in puzzle]
puzzle

###############################################################################
# PART 1

def calculate_line_of_reflection(pattern):
    R, C = pattern.shape

    LoRs = []

    # split after line r
    for r in range(1, R):
        up = pattern[: r, :]
        down = pattern[r :, :]

        rr = 0
        while rr < min(r, R - r):
            diff = abs(up[-rr - 1, :] - down[rr, :]) # compare up and down
            if sum(diff) == 0:
                rr += 1
            else:
                break
        # LoR found if running-index was able to reach an end
        if rr == min(r, R - r): LoRs.append(('r', r))

    # split after column c
    for c in range(1, C):
        left = pattern[:, : c]
        right = pattern[:, c :]

        cc = 0
        while cc < min(c, C - c):
            diff = abs(left[:, -cc - 1] - right[:, cc]) # compare left and right
            if sum(diff) == 0:
                cc += 1
            else:
                break
        # LoR found if running-index was able to reach an end
        if cc == min(c, C - c): LoRs.append(('c', c))

    return LoRs

puzzle = [{'pattern': pattern, 'LoR': calculate_line_of_reflection(pattern)} for pattern in puzzle]
assert all(len(obj['LoR']) == 1 for obj in puzzle) # assuming only one LoR
puzzle = [{'pattern': obj['pattern'], 'LoR': obj['LoR'][0]} for obj in puzzle]
puzzle

# score
S = 0
for obj in puzzle:
    m = 1 if obj['LoR'][0] == 'c' else 100
    S += (m * obj['LoR'][1])
S

###############################################################################
# PART 2

for i, obj in enumerate(puzzle):

    R, C = obj['pattern'].shape
    new_LoR = False

    # go through each row and columns
    r = 0
    while not new_LoR and r < R:
        c = 0
        while not new_LoR and c < C:
            tmp = obj['pattern'].copy()
            tmp[r, c] = 1 - tmp[r, c]

            # return LoRs which are not the original
            LoRs = calculate_line_of_reflection(tmp)
            LoRs = [LoR for LoR in LoRs if LoR != obj['LoR']]

            if LoRs != []:
                assert len(LoRs) == 1 # also assuming only one
                new_LoR = True
                obj['LoR'] = LoRs[0]

            c += 1
        r += 1

    # only finish an object if the new LoR is 100 % found
    assert new_LoR

# score
S = 0
for obj in puzzle:
    m = 1 if obj['LoR'][0] == 'c' else 100
    S += (m * obj['LoR'][1])
S